# 04 · Embeddings no supervisados

Calcula artefactos no supervisados:

- similitudes por coseno;
- vecinos semánticos;
- tópicos LDA;
- clustering KMeans/GMM;
- clustering exploratorio Agglomerative/DBSCAN;
- reducción 2D para visualización.

Este notebook no reentrena el modelo supervisado.

Metodología:
- `reference`: corpus base usado para ajustar modelos no supervisados extrapolables.
- `external`: corpus externo proyectado/predicho sobre modelos ajustados en reference.

**Nota:**
Agglomerative, DBSCAN y t-SNE se calculan como análisis exploratorio sobre el corpus combinado,
porque no son modelos ideales para inferencia directa sobre nuevos textos.

Para una visualización completa, se recomienda usar **nbviewer**:

- [04_embeddings_unsupervised.ipynb](https://nbviewer.org/github/HubertRonald/VersoVector/blob/main/notebook/04_embeddings_unsupervised.ipynb)

In [11]:
# %%
from __future__ import annotations

import sys

# Permite importar modules/ y utils/ cuando el notebook se ejecuta desde notebook/
sys.path.insert(0, "..")

%load_ext autoreload
%autoreload 2

# python
import json
import warnings

import numpy as np
import pandas as pd
from scipy import sparse

# sklearn
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.exceptions import ConvergenceWarning
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_distances

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    umap = None
    UMAP_AVAILABLE = False

# modules
from modules.clustering import (
    cosine_similarity_matrix,
    get_top_neighbors_by_cosine,
    recommend_by_cosine,
    reduce_features_dimensionality,
    fit_minibatch_kmeans_range_fast,
    fit_gmm,
    fit_lda_topics,
    transform_lda_topics,
    extract_top_words,
    topic_terms_map,
)
from modules.evaluation import cluster_metrics_from_silhouette
from modules.io import (
    artifact_path,
    load_csv,
    load_joblib,
    save_csv,
    save_json,
    save_joblib,
)
from modules.preprocessing import parse_tags

# typings
from pandas import DataFrame as PandasDF
from typing import Dict, Union

# warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore")

# setup
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("max_colwidth", None)

np.set_printoptions(precision=6)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Cargar artefactos del notebook 02

In [12]:
# Cargar feature pipeline y transformar textos
feature_pipeline_path = artifact_path("features", "feature_pipeline.joblib")

if not feature_pipeline_path.is_file():
    raise FileNotFoundError(
        f"No existe {feature_pipeline_path}. "
        "Ejecuta primero notebook/02_feature_pipeline.ipynb"
    )
    
feature_pipeline = load_joblib(feature_pipeline_path)

reference_df: PandasDF = load_csv(
    artifact_path("features", "reference_metadata.csv")
)

external_df: PandasDF = load_csv(
    artifact_path("features", "external_metadata.csv")
)

reference_df["tags"] = reference_df["tags"].apply(parse_tags)

if not external_df.empty and "tags" in external_df.columns:
    external_df["tags"] = external_df["tags"].apply(parse_tags)

print("reference_df:", reference_df.shape)
print("external_df:", external_df.shape)

display(reference_df.head(2))
display(external_df.head(2))

reference_df: (13747, 10)
external_df: (4, 10)


,title,title_raw,poet,poet_raw,poem,tags,source,corpus_role,poem_raw,poem_processed
0,objects used to prop open a window,\r\r\n Objects Used to Prop Open a Window\r\r\n,michelle menting,Michelle Menting,dog bone stapler cribbage board garlic press because this window is looselacks suction lacks grip bungee cord bootstrap dog leash leather belt because this window had sash cords they frayed they broke feather duster thatch of straw empty bottle of elmers glue because this window is loudits hinges clack open clack shut stuffed bear baby blanket single crib newel because this window is split its dividing in two velvet moss sagebrush willow branch robins wing because this window its paneless its only a frame of air,[],poetry_foundation,reference,"\r\r\nDog bone, stapler,\r\r\ncribbage board, garlic press\r\r\n because this window is loose—lacks\r\r\nsuction, lacks grip.\r\r\nBungee cord, bootstrap,\r\r\ndog leash, leather belt\r\r\n because this window had sash cords.\r\r\nThey frayed. They broke.\r\r\nFeather duster, thatch of straw, empty\r\r\nbottle of Elmer's glue\r\r\n because this window is loud—its hinges clack\r\r\nopen, clack shut.\r\r\nStuffed bear, baby blanket,\r\r\nsingle crib newel\r\r\n because this window is split. It's dividing\r\r\nin two.\r\r\nVelvet moss, sagebrush,\r\r\nwillow branch, robin's wing\r\r\n because this window, it's pane-less. It's only\r\r\na frame of air.\r\r\n",dog bone stapler cribbag board garlic press window looselack suction lack grip bunge cord bootstrap dog leash leather belt window sash cord fray break feather duster thatch straw bottl elmer glue window loudit he clack open clack shut stuf bear babi blanket singl crib newel window split divid velvet moss sagebrush willow branch robin wing window paneless frame air
1,the new church,\r\r\n The New Church\r\r\n,lucia cherciu,Lucia Cherciu,the old cupola glinted above the clouds shone among fir trees but it took him an hour for the half mile all the way up the hill as he trailed the village passed him by greeted him asked about his health but everybody hurried to catch the mass left him leaning against fences measuring the road with the walking stick he sculpted he yearned for the day when the new church would be builtright across the road now it rises above the moon saints in frescoes meet the eye and only the rain has started to cut through the shingles on the roof of his empty house the apple trees have taken over the sky sequestered the gate sidled over the porch,[],poetry_foundation,reference,"\r\r\nThe old cupola glinted above the clouds, shone\r\r\namong fir trees, but it took him an hour\r\r\nfor the half mile all the way up the hill. As he trailed,\r\r\nthe village passed him by, greeted him,\r\r\nasked about his health, but everybody hurried\r\r\nto catch the mass, left him leaning against fences,\r\r\nmeasuring the road with the walking stick he sculpted.\r\r\nHe yearned for the day when the new church\r\r\nwould be built—right across the road. Now\r\r\nit rises above the moon: saints in frescoes\r\r\nmeet the eye, and only the rain has started to cut\r\r\nthrough the shingles on the roof of his empty\r\r\nhouse. The apple trees have taken over the sky,\r\r\nsequestered the gate, sidled over the porch.\r\r\n",old cupola glint cloud shone fir tree take hour half mile way hill trail villag pass greet ask health everybodi hurri catch mass leave lean fenc measur road walk stick sculpt yearn day new church builtright road rise moon saint fresco meet eye rain start cut shingl roof hous appl tree take sky sequest gate sidl porch


,title,title_raw,poet,poet_raw,poem,tags,source,corpus_role,poem_raw,poem_processed
0,the black heralds,The Black Heralds,cesar vallejo,César Vallejo,there are blows in life so powerful i dont know blows as from gods hatred as if before them the backlash of everything suffered were to dam up in the soul i dont know they are few but they are they open dark furrows in the fiercest face and in the strongest side maybe they could be the horses of barbarous attilas or the black heralds death sends us they are the deep abysses of the souls christs of some revered faith destiny blasphemes those gory blows are the cracklings of a bread that burnsup on us at the ovens door and man poor poor he turns his eyes as when a slap on the shoulder calls us he turns his crazed eyes and everything lived is dammed up like a pond of guilt in his gaze there are blows in life so powerful i dont know,[],cesar_vallejo,external,"There are blows in life, so powerful... I don't know!\nBlows as from God's hatred; as if before them,\nthe backlash of everything suffered\nwere to dam up in the soul... I don't know!\n\nThey are few; but they are . . . They open dark furrows\nin the fiercest face and in the strongest side.\nMaybe they could be the horses of barbarous Attilas;\nor the black heralds Death sends us.\n\nThey are the deep abysses of the soul's Christs,\nof some revered faith Destiny blasphemes.\nThose gory blows are the cracklings of a bread\nthat burns-up on us at the oven's door.\n\nAnd man . . . Poor . . . poor! He turns his eyes,\nas when a slap on the shoulder calls us;\nhe turns his crazed eyes, and everything lived\nis dammed up, like a pond of guilt, in his gaze.\nThere are blows in life, so powerful . . . I don't know!",blow life power not know blow god hatr backlash suffer dam soul not know open dark furrow fierce face strong mayb hors barbar attila black herald death send deep abyss soul christ rever faith destini blasphem gori blow crackl bread burnsup oven door man poor poor turn eye slap shoulder call turn craze eye live dam like pond guilt gaze blow life power not know
1,black stone on top of a white stone,Black Stone On Top Of A White Stone,cesar vallejo,César Vallejo,i shall die in paris in a rainstorm on a day i already remember i shall die in paris it does not bother me doubtless on a thursday like today in autumn it shall be a thursday because today thursday as i put down these lines i have set my shoulders to the evil never like today have i turned and headed my whole journey to the ways where i am alone cesar vallejo is dead they struck him all of them though he did nothing to them they hit him hard with a stick and hard also with the end of a rope witnesses are the thursdays the shoulder bones the loneliness the rain and the roads,[],cesar_vallejo,external,"I shall die in Paris, in a rainstorm,\nOn a day I already remember.\nI shall die in Paris- it does not bother me-\nDoubtless on a Thursday, like today, in autumn.\n\nIt shall be a Thursday, because today, Thursday\nAs I put down these lines, I have set my shoulders\nTo the evil. Never like today have I turned,\nAnd headed my whole journey to the ways where I am alone.\n\nCésar Vallejo is dead. They struck him,\nAll of them, though he did nothing to them,\nThey hit him hard with a stick and hard also\nWith the end of a rope. Witnesses are: the Thursdays,\nThe shoulder bones, the loneliness, the rain, and the roads...",shall die pari rainstorm day rememb shall die pari bother doubtless thursday like today autumn shall thursday today thursday line set shoulder evil like today turn head journey way cesar vallejo dead strike hit hard stick hard end rope wit thursday shoulder bone loneli rain road


## 2. Transformar textos con el pipeline ajustado en reference

El pipeline fue ajustado en el notebook 02.
Aquí solo se usa `transform`.

In [13]:
reference_texts = reference_df["poem_processed"].astype(str).tolist()
external_texts = external_df["poem_processed"].astype(str).tolist()

X_reference = feature_pipeline.transform(reference_texts)

if external_texts:
    X_external = feature_pipeline.transform(external_texts)
else:
    X_external = sparse.csr_matrix((0, X_reference.shape[1]))

print("X_reference shape:", X_reference.shape)
print("X_external shape:", X_external.shape)

X_reference shape: (13747, 223671)
X_external shape: (4, 223671)


In [14]:
if X_external.shape[0] > 0:
    X_all = sparse.vstack([X_reference, X_external])
    all_metadata = pd.concat(
        [reference_df, external_df],
        ignore_index=True,
    )
else:
    X_all = X_reference
    all_metadata = reference_df.copy()

all_metadata = all_metadata.reset_index(drop=True)

print("X_all shape:", X_all.shape)
print("all_metadata:", all_metadata.shape)

X_all shape: (13751, 223671)
all_metadata: (13751, 10)


## 3. Similitud coseno y vecinos semánticos

Se calcula sobre `X_all`, pero el espacio fue aprendido desde reference.

In [15]:
cosine_sim = cosine_similarity_matrix(X_all)

query_title = "the black heralds"

if query_title in set(all_metadata["title"]):
    display(
        recommend_by_cosine(
            query_title,
            cosine_sim=cosine_sim,
            df=all_metadata,
            top_n=10,
        )
    )
else:
    print(f"No existe '{query_title}' en el corpus actual.")

nearest_titles_cosine, nearest_scores_cosine = get_top_neighbors_by_cosine(
    cosine_sim=cosine_sim,
    df=all_metadata,
    top_n=5,
)

,puesto,title,poet,source,corpus_role,similarity_cosine
0,1,the aim was song,robert frost,poetry_foundation,reference,0.225006
1,2,ballad of the salvation army,kenneth fearing,poetry_foundation,reference,0.207743
2,3,keumganggul diamond cave,ko un,poetry_foundation,reference,0.205286
3,4,fortuna,thomas carlyle,poetry_foundation,reference,0.202982
4,5,which way the winds blow,alice b fogel,poetry_foundation,reference,0.200919
5,6,from the princess the splendour falls on castle walls,alfred lord tennyson,poetry_foundation,reference,0.188566
6,7,mans short life and foolish ambition,duchess of newcastle margaret cavendish,poetry_foundation,reference,0.181937
7,8,in memoriam a h h obiit mdcccxxxiii,alfred lord tennyson,poetry_foundation,reference,0.178609
8,9,from the book of the dead the dam,muriel rukeyser,poetry_foundation,reference,0.177324
9,10,curse two the naming,cynthia huntington,poetry_foundation,reference,0.175132


## 4. Modelado de tópicos LDA

LDA se ajusta solo sobre el corpus `reference`.
Luego se transforman los poemas externos sobre ese mismo espacio de tópicos.

In [16]:
lda_model, topic_vectorizer, X_topics_reference, lda_topics_reference = fit_lda_topics(
    reference_texts,
    n_components=7,
    max_features=5000,
    random_state=42,
)

if external_texts:
    X_topics_external, lda_topics_external = transform_lda_topics(
        lda_model,
        topic_vectorizer,
        external_texts,
    )

    lda_topics = np.vstack([
        lda_topics_reference,
        lda_topics_external,
    ])
else:
    X_topics_external = None
    lda_topics = lda_topics_reference

top_words_df = extract_top_words(
    lda_model,
    topic_vectorizer,
    n_top=10,
)

topic_terms = topic_terms_map(
    top_words_df,
    n_terms=5,
)

display(top_words_df)

,topic_id,top_words,top_words_list
0,0,"man, god, good, great, shall, king, lord, ye, let, thing","[man, god, good, great, shall, king, lord, ye, let, thing]"
1,1,"like, tree, white, water, blue, green, black, come, sun, leav","[like, tree, white, water, blue, green, black, come, sun, leav]"
2,2,"come, like, night, day, eye, heart, love, light, wind, hear","[come, like, night, day, eye, heart, love, light, wind, hear]"
3,3,"thi, thou, love, thee, shall, heart, life, eye, soul, know","[thi, thou, love, thee, shall, heart, life, eye, soul, know]"
4,4,"like, know, say, think, time, want, look, way, come, feel","[like, know, say, think, time, want, look, way, come, feel]"
5,5,"like, light, bodi, hand, eye, dark, night, water, know, come","[like, light, bodi, hand, eye, dark, night, water, know, come]"
6,6,"man, like, say, know, love, come, child, die, mother, tell","[man, like, say, know, love, come, child, die, mother, tell]"


## 5. Reducción dimensional para clustering

Si `X_reference` es sparse, `reduce_features_pca` usa internamente `TruncatedSVD`.
Si es dense, usa `PCA`.

In [17]:
X_reference_cluster, cluster_reducer = reduce_features_dimensionality(
    X_reference,
    n_components=100,
    random_state=42,
)

if X_external.shape[0] > 0:
    X_external_cluster = cluster_reducer.transform(X_external)
    X_cluster_all = np.vstack([
        X_reference_cluster,
        X_external_cluster,
    ])
else:
    X_external_cluster = np.empty((0, X_reference_cluster.shape[1]))
    X_cluster_all = X_reference_cluster

print("X_reference_cluster:", X_reference_cluster.shape)
print("X_external_cluster:", X_external_cluster.shape)
print("X_cluster_all:", X_cluster_all.shape)

X_reference_cluster: (13747, 100)
X_external_cluster: (4, 100)
X_cluster_all: (13751, 100)


## 6. KMeans extrapolable

`MiniBatchKMeans` se ajusta sobre `reference`.
Luego se predicen clusters para `external`.

In [18]:
best_km, km_sils = fit_minibatch_kmeans_range_fast(
    X_reference_cluster,
    kmin=2,
    kmax=8,
    random_state=42,
    sample_size=1000,
    batch_size=1024,
)

reference_cluster_km = best_km["labels"]

if X_external_cluster.shape[0] > 0:
    external_cluster_km = best_km["model"].predict(X_external_cluster)
    cluster_km = np.concatenate([
        reference_cluster_km,
        external_cluster_km,
    ])
else:
    cluster_km = reference_cluster_km

cluster_metrics_df = cluster_metrics_from_silhouette(
    km_sils,
    model_name="MiniBatchKMeans",
)

display(cluster_metrics_df)
print("best_k:", best_km["k"])
print("best_silhouette:", best_km["sil"])

,model,k,silhouette
0,MiniBatchKMeans,2,0.056583
1,MiniBatchKMeans,3,-0.016036
2,MiniBatchKMeans,4,0.002406
3,MiniBatchKMeans,5,-0.000795
4,MiniBatchKMeans,6,0.001334
5,MiniBatchKMeans,7,0.008603
6,MiniBatchKMeans,8,0.000608


best_k: 2
best_silhouette: 0.05658280136629981


## 7. GMM extrapolable

`GaussianMixture` se ajusta sobre `reference`.
Luego se predicen clusters para `external`.

In [19]:
gmm_model, reference_cluster_gmm = fit_gmm(
    X_reference_cluster,
    n_components=best_km["k"],
    covariance_type="diag",
    random_state=42,
)

if X_external_cluster.shape[0] > 0:
    external_cluster_gmm = gmm_model.predict(X_external_cluster)
    cluster_gmm = np.concatenate([
        reference_cluster_gmm,
        external_cluster_gmm,
    ])
else:
    cluster_gmm = reference_cluster_gmm

## 8. Clustering exploratorio combinado

Agglomerative y DBSCAN se calculan sobre el corpus combinado.
Estos resultados son útiles para exploración, pero no se consideran modelos extrapolables.

In [20]:
D = cosine_distances(X_cluster_all)

agg = AgglomerativeClustering(
    n_clusters=best_km["k"],
    metric="precomputed",
    linkage="average",
)

cluster_agg = agg.fit_predict(D)

## 9. Reducción 2D exploratoria

In [21]:
if UMAP_AVAILABLE:
    try:
        reducer = umap.UMAP(
            n_neighbors=15,
            min_dist=0.1,
            random_state=42,
        )
        X2 = reducer.fit_transform(X_cluster_all)
        reducer_name = "UMAP_combined"
    except Exception as exc:
        print(f"UMAP falló, se usará t-SNE. Detalle: {exc}")
        X2 = TSNE(
            n_components=2,
            random_state=42,
            metric="cosine",
        ).fit_transform(X_cluster_all)
        reducer_name = "t-SNE_combined"
else:
    print("UMAP no está instalado. Se usará t-SNE.")
    X2 = TSNE(
        n_components=2,
        random_state=42,
        metric="cosine",
    ).fit_transform(X_cluster_all)
    reducer_name = "t-SNE_combined"

cluster_dbscan = DBSCAN(
    eps=0.5,
    min_samples=5,
    metric="euclidean",
).fit_predict(X2)

print("reducer_name:", reducer_name)

UMAP no está instalado. Se usará t-SNE.
reducer_name: t-SNE_combined


## 10. Construir resultados no supervisados

In [22]:
lda_topic_id = lda_topics.argmax(axis=1)
lda_topic_prob = lda_topics.max(axis=1)

unsupervised_results = pd.DataFrame({
    "title": all_metadata["title"].values,
    "title_raw": all_metadata["title_raw"].values,
    "poet": all_metadata["poet"].values,
    "poet_raw": all_metadata["poet_raw"].values,
    "source": all_metadata["source"].values,
    "corpus_role": all_metadata["corpus_role"].values,

    "cluster_km": cluster_km,
    "cluster_gmm": cluster_gmm,

    # Exploratorios combinados
    "cluster_agg": cluster_agg,
    "cluster_dbscan": cluster_dbscan,

    "lda_topic_id": lda_topic_id,
    "lda_topic_prob": np.round(lda_topic_prob, 4),
    "lda_topic_terms": [
        topic_terms.get(int(topic_id), "")
        for topic_id in lda_topic_id
    ],

    "nearest_titles_cosine_json": [
        json.dumps(titles, ensure_ascii=False)
        for titles in nearest_titles_cosine
    ],
    "nearest_scores_cosine_json": [
        json.dumps(scores, ensure_ascii=False)
        for scores in nearest_scores_cosine
    ],

    "embedding_x": X2[:, 0],
    "embedding_y": X2[:, 1],
})

embeddings_2d = unsupervised_results[
    [
        "title",
        "poet",
        "source",
        "corpus_role",
        "embedding_x",
        "embedding_y",
        "cluster_km",
        "cluster_gmm",
        "cluster_agg",
        "cluster_dbscan",
    ]
].copy()

display(unsupervised_results.head(10))

,title,title_raw,poet,poet_raw,source,corpus_role,cluster_km,cluster_gmm,cluster_agg,cluster_dbscan,lda_topic_id,lda_topic_prob,lda_topic_terms,nearest_titles_cosine_json,nearest_scores_cosine_json,embedding_x,embedding_y
0,objects used to prop open a window,\r\r\n Objects Used to Prop Open a Window\r\r\n,michelle menting,Michelle Menting,poetry_foundation,reference,0,1,0,-1,1,0.6896,"like, tree, white, water, blue","[""confession of a bird watcher"", ""mother and child body and soul"", ""supple cord"", ""schizotableau"", ""winter""]","[0.1981, 0.1817, 0.1762, 0.1747, 0.173]",-25.025835,-55.625130
1,the new church,\r\r\n The New Church\r\r\n,lucia cherciu,Lucia Cherciu,poetry_foundation,reference,0,0,0,-1,1,0.6734,"like, tree, white, water, blue","[""uppity"", ""drifting at midday"", ""what about this"", ""photo of a man on sunset drive"", ""perseus in arkansas""]","[0.2142, 0.2016, 0.1951, 0.1939, 0.1926]",-39.418159,-14.665663
2,look for me,\r\r\n Look for Me\r\r\n,ted kooser,Ted Kooser,poetry_foundation,reference,0,1,0,-1,1,0.5274,"like, tree, white, water, blue","[""this is the time of grasshoppers and all that i see is dying"", ""ode to a grasshopper"", ""radiator"", ""pink slip at tool dye"", ""moths""]","[0.177, 0.1704, 0.1601, 0.1521, 0.147]",11.992420,32.972317
3,wild life,\r\r\n Wild Life\r\r\n,grace cavalieri,Grace Cavalieri,poetry_foundation,reference,0,1,0,9,1,0.4714,"like, tree, white, water, blue","[""darwins bestiary"", ""a reallife drama"", ""the dream"", ""the springtime"", ""the relics""]","[0.197, 0.1722, 0.1716, 0.1608, 0.1418]",-57.405487,-8.228206
4,umbrella,\r\r\n Umbrella\r\r\n,connie wanek,Connie Wanek,poetry_foundation,reference,0,0,0,-1,5,0.8986,"like, light, bodi, hand, eye","[""bat"", ""here i am lord"", ""sestina like"", ""english mole"", ""the last skin""]","[0.2027, 0.1872, 0.1864, 0.1709, 0.1656]",0.147400,-17.399794
5,sunday,\r\r\n Sunday\r\r\n,january gill oneil,January Gill O'Neil,poetry_foundation,reference,0,0,0,0,4,0.4982,"like, know, say, think, time","[""a poem called day"", ""day after day of the dead"", ""the troubadours etc"", ""night magic blue jester"", ""lakes rivers streams""]","[0.2509, 0.2202, 0.2186, 0.2128, 0.2031]",-0.280615,-19.742186
6,invisible fish,\r\r\n Invisible Fish\r\r\n,joy harjo,Joy Harjo,poetry_foundation,reference,0,1,0,-1,5,0.9703,"like, light, bodi, hand, eye","[""salt"", ""future memories"", ""master class"", ""silver and information"", ""in the aquarium""]","[0.2562, 0.2341, 0.2204, 0.2195, 0.2158]",30.996531,-20.820089
7,dont bother the earth spirit,\r\r\n Don’t Bother the Earth Spirit\r\r\n,joy harjo,Joy Harjo,poetry_foundation,reference,0,1,0,-1,6,0.9767,"man, like, say, know, love","[""khaleesi says"", ""duty"", ""in memory of the unknown poet robert boardman vaughn"", ""lethargy"", ""the sandhills""]","[0.3613, 0.3064, 0.2823, 0.2323, 0.2315]",27.383593,44.275291
8,the one thing that can save america,\r\r\n The One Thing That Can Save America\r\r\n,john ashbery,John Ashbery,poetry_foundation,reference,0,1,0,-1,4,0.6133,"like, know, say, think, time","[""song of myself version"", ""catalog of unabashed gratitude"", ""what i know"", ""north point north"", ""falling water""]","[0.221, 0.2036, 0.1947, 0.1917, 0.1885]",-14.298364,24.975334
9,hour in which i consider hydrangea,"\r\r\n [""Hour in which I consider hydrangea""]\r\r\n",simone white,Simone White,poetry_foundation,reference,0,0,0,-1,5,0.6336,"like, light, bodi, hand, eye","[""being serious"", ""beautiful signor"", ""the bodies"", ""a prayer for my daughter"", ""staying quiet""]","[0.208, 0.2068, 0.2003, 0.198, 0.1968]",-54.357067,-6.578418


## 11. Metadata de features

Se guarda un resumen del pipeline y las dimensiones generadas.

In [23]:
unsupervised_metadata: Dict[str, Union[str,int, bool]] = {
    "feature_pipeline_artifact": "artifacts/features/feature_pipeline.joblib",
    "input_column": "poem_processed",
    "reference_fit": True,
    "external_transform": True,
    "feature_pipeline_expected": (
        "build_feature_pipeline(input_is_processed=True, "
        "to_dense=False, normalize=True)"
    ),
    "n_reference_documents": int(reference_df.shape[0]),
    "n_external_documents": int(external_df.shape[0]),
    "n_total_documents": int(all_metadata.shape[0]),
    "n_features": int(X_reference.shape[1]),
    "n_cluster_components": int(X_reference_cluster.shape[1]),
    "best_kmeans_k": int(best_km["k"]),
    "best_kmeans_silhouette": float(best_km["sil"]),
    "lda_n_topics": int(lda_model.n_components),
    "reducer_name": reducer_name,
    "agglomerative_scope": "combined_exploratory",
    "dbscan_scope": "combined_exploratory",
}


save_json(
    unsupervised_metadata,
    artifact_path("unsupervised", "unsupervised_metadata.json"),
)

unsupervised_metadata

{'feature_pipeline_artifact': 'artifacts/features/feature_pipeline.joblib',
 'input_column': 'poem_processed',
 'reference_fit': True,
 'external_transform': True,
 'feature_pipeline_expected': 'build_feature_pipeline(input_is_processed=True, to_dense=False, normalize=True)',
 'n_reference_documents': 13747,
 'n_external_documents': 4,
 'n_total_documents': 13751,
 'n_features': 223671,
 'n_cluster_components': 100,
 'best_kmeans_k': 2,
 'best_kmeans_silhouette': 0.05658280136629981,
 'lda_n_topics': 7,
 'reducer_name': 't-SNE_combined',
 'agglomerative_scope': 'combined_exploratory',
 'dbscan_scope': 'combined_exploratory'}

## 12. Guardar artefactos no supervisados

In [24]:
# %%
save_csv(
    unsupervised_results,
    artifact_path("unsupervised", "unsupervised_results.csv"),
)

save_csv(
    top_words_df.drop(columns=["top_words_list"], errors="ignore"),
    artifact_path("unsupervised", "lda_topics.csv"),
)

save_csv(
    cluster_metrics_df,
    artifact_path("unsupervised", "cluster_metrics.csv"),
)

save_csv(
    embeddings_2d,
    artifact_path("unsupervised", "embeddings_2d.csv"),
)

save_json(
    {"reducer_name": reducer_name},
    artifact_path("unsupervised", "reducer_metadata.json"),
)

# Opcional: guardar modelos pequeños/medianos locales.
# No subir .joblib a GitHub.
save_joblib(
    best_km["model"],
    artifact_path("unsupervised", "kmeans_model.joblib"),
)

save_joblib(
    gmm_model,
    artifact_path("unsupervised", "gmm_model.joblib"),
)

save_joblib(
    lda_model,
    artifact_path("unsupervised", "lda_model.joblib"),
)

save_joblib(
    topic_vectorizer,
    artifact_path("unsupervised", "lda_count_vectorizer.joblib"),
)

print("Artefactos no supervisados guardados en artifacts/unsupervised/")

Artefactos no supervisados guardados en artifacts/unsupervised/
